# Tissue Ontology Coverage Analysis

This notebook analyzes:
1. Which tissue ontology IDs from training/test data are in CellxGene's curated vocabulary
2. Which tissues are missing from the vocabulary
3. Whether the TissueEncoder is loading the ontology correctly
4. Impact on training and test data filtering

In [28]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
import duckdb

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_loading.tissue_encoder import TissueEncoder

## 1. Load TissueEncoder and Inspect Vocabulary

In [29]:
# Initialize encoder
encoder = TissueEncoder()

print("TissueEncoder Vocabulary:")
print(f"  Tissues: {len(encoder.tissue_to_idx)}")
print(f"  Organs: {len(encoder.organ_to_idx)}")
print(f"  Systems: {len(encoder.system_to_idx)}")
print(f"  Total dimensions: {encoder.total_dim}")

# Show sample tissues
print("\nSample tissues (first 20):")
for i, tissue_id in enumerate(sorted(encoder.tissue_to_idx.keys())[:20]):
    print(f"  {tissue_id}")

TissueEncoder Vocabulary:
  Tissues: 81
  Organs: 28
  Systems: 17
  Total dimensions: 126

Sample tissues (first 20):
  UBERON:0000004
  UBERON:0000010
  UBERON:0000014
  UBERON:0000029
  UBERON:0000030
  UBERON:0000056
  UBERON:0000057
  UBERON:0000059
  UBERON:0000160
  UBERON:0000175
  UBERON:0000178
  UBERON:0000310
  UBERON:0000344
  UBERON:0000383
  UBERON:0000473
  UBERON:0000916
  UBERON:0000922
  UBERON:0000945
  UBERON:0000948
  UBERON:0000949


## 2. Verify Ontology Loading

Check if the ontology parser and curated lists are being loaded correctly.

In [30]:
# Check what's in the curated lists
print("Curated Lists from CellxGene:")
print(f"  curated_tissues: {len(encoder.curated_tissues)} terms")
print(f"  curated_organs: {len(encoder.curated_organs)} terms")
print(f"  curated_systems: {len(encoder.curated_systems)} terms")

# Show samples
print("\nSample curated tissues:")
for tissue in list(encoder.curated_tissues)[:10]:
    print(f"  {tissue}")

print("\nSample curated organs:")
for organ in list(encoder.curated_organs)[:10]:
    print(f"  {organ}")

print("\nSample curated systems:")
for system in list(encoder.curated_systems)[:10]:
    print(f"  {system}")

Curated Lists from CellxGene:
  curated_tissues: 81 terms
  curated_organs: 28 terms
  curated_systems: 17 terms

Sample curated tissues:
  UBERON:0000178
  UBERON:0002048
  UBERON:0002106
  UBERON:0002371
  UBERON:0002107
  UBERON:0002113
  UBERON:0000955
  UBERON:0002240
  UBERON:0000310
  UBERON:0000948

Sample curated organs:
  UBERON:0000992
  UBERON:0000029
  UBERON:0002048
  UBERON:0002110
  UBERON:0001043
  UBERON:0003889
  UBERON:0018707
  UBERON:0000178
  UBERON:0002371
  UBERON:0000955

Sample curated systems:
  UBERON:0001017
  UBERON:0004535
  UBERON:0001009
  UBERON:0001007
  UBERON:0000922
  UBERON:0000949
  UBERON:0002330
  UBERON:0002390
  UBERON:0002405
  UBERON:0000383


In [31]:
# Check ontology parser
print("Ontology Parser:")
print(f"  Type: {type(encoder.ontology_parser)}")
print(f"  Has ontology data: {hasattr(encoder.ontology_parser, 'ontology')}")

# Test with a known tissue ID
test_id = 'UBERON:0002107'  # liver
print(f"\nTest lookup for {test_id} (liver):")
print(f"  In tissue_to_idx: {test_id in encoder.tissue_to_idx}")
if test_id in encoder.tissue_to_idx:
    print(f"  Index: {encoder.tissue_to_idx[test_id]}")
    
    # Check hierarchy
    if test_id in encoder.tissue_to_organ_indices:
        organ_indices = encoder.tissue_to_organ_indices[test_id]
        print(f"  Organ indices: {organ_indices}")
    if test_id in encoder.tissue_to_system_indices:
        system_indices = encoder.tissue_to_system_indices[test_id]
        print(f"  System indices: {system_indices}")

Ontology Parser:
  Type: <class 'cellxgene_ontology_guide.ontology_parser.OntologyParser'>
  Has ontology data: False

Test lookup for UBERON:0002107 (liver):
  In tissue_to_idx: True
  Index: 54
  Organ indices: [20]
  System indices: [3, 6, 13]


## 3. Analyze Training Data

In [32]:
# Load training data tissue ontology IDs using DuckDB
training_dir = Path('/mmc-scratch/scratch/cellxgene_v2_training_v1')

print(f"Loading tissue ontology IDs from ALL training data using DuckDB...")

# Use DuckDB to count tissue ontology terms across all parquet files
con = duckdb.connect(':memory:')

query = f"""
SELECT 
    tissue_ontology_term_id,
    COUNT(*) as cell_count
FROM read_parquet('{training_dir}/*.parquet')
GROUP BY tissue_ontology_term_id
ORDER BY cell_count DESC
"""

training_tissue_df = con.execute(query).fetchdf()
con.close()

# Convert to Counter for compatibility with rest of notebook
training_tissue_counts = Counter(dict(zip(training_tissue_df['tissue_ontology_term_id'], 
                                           training_tissue_df['cell_count'])))
training_tissues = set(training_tissue_counts.keys())
training_total_cells = training_tissue_df['cell_count'].sum()

print(f"\nTraining data (ALL {len(list(training_dir.glob('*.parquet')))} FILES):")
print(f"  Total cells: {training_total_cells:,}")
print(f"  Unique tissue ontology IDs: {len(training_tissues)}")

Loading tissue ontology IDs from ALL training data using DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Training data (ALL 643 FILES):
  Total cells: 3,769,346
  Unique tissue ontology IDs: 237


In [33]:
# Check coverage
encoder_tissues = set(encoder.tissue_to_idx.keys())

training_in_encoder = training_tissues & encoder_tissues
training_not_in_encoder = training_tissues - encoder_tissues

print("Training Data Coverage:")
print(f"  Tissues in encoder: {len(training_in_encoder)} / {len(training_tissues)} ({len(training_in_encoder)/len(training_tissues)*100:.1f}%)")
print(f"  Tissues NOT in encoder: {len(training_not_in_encoder)} ({len(training_not_in_encoder)/len(training_tissues)*100:.1f}%)")

# Count cells affected
cells_with_valid_tissue = sum(training_tissue_counts[t] for t in training_in_encoder)
cells_with_invalid_tissue = sum(training_tissue_counts[t] for t in training_not_in_encoder)

print(f"\nCell-level impact:")
print(f"  Cells with valid tissue: {cells_with_valid_tissue:,} ({cells_with_valid_tissue/training_total_cells*100:.1f}%)")
print(f"  Cells with invalid tissue: {cells_with_invalid_tissue:,} ({cells_with_invalid_tissue/training_total_cells*100:.1f}%)")

Training Data Coverage:
  Tissues in encoder: 38 / 237 (16.0%)
  Tissues NOT in encoder: 199 (84.0%)

Cell-level impact:
  Cells with valid tissue: 1,484,211 (39.4%)
  Cells with invalid tissue: 2,285,135 (60.6%)


In [34]:
# Show missing tissues from training
if len(training_not_in_encoder) > 0:
    print("\nTop 20 missing tissues in TRAINING (by cell count):")
    missing_tissues_sorted = sorted(
        [(t, training_tissue_counts[t]) for t in training_not_in_encoder],
        key=lambda x: x[1],
        reverse=True
    )
    for tissue_id, count in missing_tissues_sorted[:20]:
        print(f"  {tissue_id:30s} {count:8,} cells")
else:
    print("\n✓ All training tissues are in encoder vocabulary!")


Top 20 missing tissues in TRAINING (by cell count):
  UBERON:0002116                   95,260 cells
  UBERON:0000991                   94,837 cells
  UBERON:0001225                   86,750 cells
  UBERON:0008946                   85,032 cells
  UBERON:0001117                   75,845 cells
  UBERON:8410025                   69,604 cells
  UBERON:0002084                   63,724 cells
  UBERON:0002771                   61,606 cells
  UBERON:0001005                   58,283 cells
  UBERON:0000966                   57,592 cells
  UBERON:0001161                   47,750 cells
  UBERON:0000006                   47,407 cells
  UBERON:0001111                   46,355 cells
  UBERON:0001295                   44,461 cells
  UBERON:0013682                   41,540 cells
  UBERON:0001786                   38,203 cells
  UBERON:0009834                   36,858 cells
  UBERON:0001707                   33,510 cells
  UBERON:0000053                   33,504 cells
  UBERON:0002421                   

## 4. Analyze Test Data

In [35]:
# Load test data tissue ontology IDs using DuckDB
test_dir = Path('/mmc-scratch/scratch/cellxgene_v2_test_v1')

print(f"Loading tissue ontology IDs from test data using DuckDB...")

# Use DuckDB to count tissue ontology terms across all parquet files
con = duckdb.connect(':memory:')

query = f"""
SELECT 
    tissue_ontology_term_id,
    COUNT(*) as cell_count
FROM read_parquet('{test_dir}/*.parquet')
GROUP BY tissue_ontology_term_id
ORDER BY cell_count DESC
"""

test_tissue_df = con.execute(query).fetchdf()
con.close()

# Convert to Counter for compatibility with rest of notebook
test_tissue_counts = Counter(dict(zip(test_tissue_df['tissue_ontology_term_id'], 
                                       test_tissue_df['cell_count'])))
test_tissues = set(test_tissue_counts.keys())
test_total_cells = test_tissue_df['cell_count'].sum()

# Also store test_files for later use
test_files = sorted(test_dir.glob('*.parquet'))

print(f"\nTest data (ALL {len(test_files)} FILES):")
print(f"  Total cells: {test_total_cells:,}")
print(f"  Unique tissue ontology IDs: {len(test_tissues)}")

Loading tissue ontology IDs from test data using DuckDB...



Test data (ALL 15 FILES):
  Total cells: 120,984
  Unique tissue ontology IDs: 17


In [36]:
# Check coverage
test_in_encoder = test_tissues & encoder_tissues
test_not_in_encoder = test_tissues - encoder_tissues

print("Test Data Coverage:")
print(f"  Tissues in encoder: {len(test_in_encoder)} / {len(test_tissues)} ({len(test_in_encoder)/len(test_tissues)*100:.1f}%)")
print(f"  Tissues NOT in encoder: {len(test_not_in_encoder)} ({len(test_not_in_encoder)/len(test_tissues)*100:.1f}%)")

# Count cells affected
cells_with_valid_tissue = sum(test_tissue_counts[t] for t in test_in_encoder)
cells_with_invalid_tissue = sum(test_tissue_counts[t] for t in test_not_in_encoder)

print(f"\nCell-level impact:")
print(f"  Cells with valid tissue: {cells_with_valid_tissue:,} ({cells_with_valid_tissue/test_total_cells*100:.1f}%)")
print(f"  Cells with invalid tissue: {cells_with_invalid_tissue:,} ({cells_with_invalid_tissue/test_total_cells*100:.1f}%)")

Test Data Coverage:
  Tissues in encoder: 5 / 17 (29.4%)
  Tissues NOT in encoder: 12 (70.6%)

Cell-level impact:
  Cells with valid tissue: 50,206 (41.5%)
  Cells with invalid tissue: 70,778 (58.5%)


In [37]:
# Show missing tissues from test
if len(test_not_in_encoder) > 0:
    print("\nMissing tissues in TEST (by cell count):")
    missing_tissues_sorted = sorted(
        [(t, test_tissue_counts[t]) for t in test_not_in_encoder],
        key=lambda x: x[1],
        reverse=True
    )
    for tissue_id, count in missing_tissues_sorted:
        # Try to get tissue name
        sample_file = next(test_dir.glob('*.parquet'))
        df_sample = pd.read_parquet(sample_file, columns=['tissue_ontology_term_id', 'tissue'])
        name_row = df_sample[df_sample['tissue_ontology_term_id'] == tissue_id]
        if len(name_row) > 0:
            tissue_name = name_row['tissue'].iloc[0]
        else:
            tissue_name = "(name not found)"
        
        print(f"  {tissue_id:30s} {count:8,} cells - {tissue_name}")
else:
    print("\n✓ All test tissues are in encoder vocabulary!")


Missing tissues in TEST (by cell count):


  UBERON:0002686                   22,033 cells - (name not found)
  UBERON:0001111                   11,806 cells - (name not found)
  UBERON:8480009                    8,991 cells - tendon of semitendinosus
  UBERON:0014614                    5,347 cells - (name not found)
  UBERON:0002728                    4,991 cells - (name not found)
  UBERON:0002317                    4,974 cells - (name not found)
  UBERON:0013535                    3,364 cells - (name not found)
  UBERON:0002185                    3,103 cells - (name not found)
  UBERON:0008952                    2,170 cells - (name not found)
  UBERON:0008953                    1,765 cells - (name not found)
  UBERON:0003544                    1,517 cells - (name not found)
  UBERON:0003126                      717 cells - (name not found)


## 5. Compare Training vs Test Coverage

In [38]:
# Venn diagram style comparison
only_in_training = training_not_in_encoder - test_not_in_encoder
only_in_test = test_not_in_encoder - training_not_in_encoder
missing_in_both = training_not_in_encoder & test_not_in_encoder

print("Missing Tissue Comparison:")
print(f"  Missing ONLY in training: {len(only_in_training)}")
print(f"  Missing ONLY in test: {len(only_in_test)}")
print(f"  Missing in BOTH: {len(missing_in_both)}")

if len(only_in_test) > 0:
    print("\nTissues missing ONLY in test (problematic for evaluation):")
    for tissue_id in sorted(only_in_test):
        count = test_tissue_counts[tissue_id]
        print(f"  {tissue_id:30s} {count:8,} cells")

Missing Tissue Comparison:
  Missing ONLY in training: 189
  Missing ONLY in test: 2
  Missing in BOTH: 10

Tissues missing ONLY in test (problematic for evaluation):
  UBERON:0003544                    1,517 cells
  UBERON:8480009                    8,991 cells


## 6. Summary and Recommendations

In [39]:
print("=" * 100)
print("SUMMARY")
print("=" * 100)

print("\nEncoder Vocabulary:")
print(f"  Total tissues: {len(encoder_tissues)}")
print(f"  Dimensions: {encoder.total_dim} (81 tissue + 28 organ + 17 system)")

print("\nTraining Data (sampled):")
print(f"  Unique tissues: {len(training_tissues)}")
print(f"  Coverage: {len(training_in_encoder)}/{len(training_tissues)} ({len(training_in_encoder)/len(training_tissues)*100:.1f}%)")
print(f"  Cells with valid tissue: {cells_with_valid_tissue:,} ({cells_with_valid_tissue/training_total_cells*100:.1f}%)")

print("\nTest Data:")
test_valid = sum(test_tissue_counts[t] for t in test_in_encoder)
test_invalid = sum(test_tissue_counts[t] for t in test_not_in_encoder)
print(f"  Unique tissues: {len(test_tissues)}")
print(f"  Coverage: {len(test_in_encoder)}/{len(test_tissues)} ({len(test_in_encoder)/len(test_tissues)*100:.1f}%)")
print(f"  Cells with valid tissue: {test_valid:,} ({test_valid/test_total_cells*100:.1f}%)")
print(f"  Cells with invalid tissue: {test_invalid:,} ({test_invalid/test_total_cells*100:.1f}%)")

print("\n" + "=" * 100)
print("RECOMMENDATIONS")
print("=" * 100)

if test_invalid > 0:
    pct_invalid = test_invalid / test_total_cells * 100
    print(f"\n⚠️  {pct_invalid:.1f}% of test cells have tissues not in the encoder vocabulary")
    print("\nOptions:")
    print("  A. Filter these cells during evaluation (track_invalid_embeddings=True)")
    print("     - Ensures model sees consistent data distribution")
    print(f"     - Evaluate on {test_valid:,} cells ({test_valid/test_total_cells*100:.1f}%)")
    print("\n  B. Don't filter (track_invalid_embeddings=False)")
    print("     - Keep all cells but with zero tissue vectors for unknown tissues")
    print(f"     - Evaluate on {test_total_cells:,} cells (100%)")
    print("     - May degrade performance due to missing features")
    print("\n  C. Remove tissue embeddings entirely")
    print("     - Change embedding_types to ['genept', 'metadata']")
    print("     - Simpler but loses tissue information")
    print("\n  D. Expand encoder vocabulary to include missing tissues")
    print("     - Requires modifying TissueEncoder or CellxGene curated lists")
    print("     - Most complete but requires development work")
else:
    print("\n✓ All test tissues are covered by the encoder vocabulary!")
    print("  No action needed for tissue encoding.")

SUMMARY

Encoder Vocabulary:
  Total tissues: 81
  Dimensions: 126 (81 tissue + 28 organ + 17 system)

Training Data (sampled):
  Unique tissues: 237
  Coverage: 38/237 (16.0%)
  Cells with valid tissue: 50,206 (1.3%)

Test Data:
  Unique tissues: 17
  Coverage: 5/17 (29.4%)
  Cells with valid tissue: 50,206 (41.5%)
  Cells with invalid tissue: 70,778 (58.5%)

RECOMMENDATIONS

⚠️  58.5% of test cells have tissues not in the encoder vocabulary

Options:
  A. Filter these cells during evaluation (track_invalid_embeddings=True)
     - Ensures model sees consistent data distribution
     - Evaluate on 50,206 cells (41.5%)

  B. Don't filter (track_invalid_embeddings=False)
     - Keep all cells but with zero tissue vectors for unknown tissues
     - Evaluate on 120,984 cells (100%)
     - May degrade performance due to missing features

  C. Remove tissue embeddings entirely
     - Change embedding_types to ['genept', 'metadata']
     - Simpler but loses tissue information

  D. Expand enc

In [40]:
# Create DataFrame of missing test tissues with names
if len(test_not_in_encoder) > 0:
    missing_data = []
    
    for pq_file in test_files:
        df = pd.read_parquet(pq_file, columns=['tissue_ontology_term_id', 'tissue'])
        for tissue_id in test_not_in_encoder:
            matches = df[df['tissue_ontology_term_id'] == tissue_id]
            if len(matches) > 0:
                tissue_name = matches['tissue'].iloc[0]
                if not any(d['tissue_id'] == tissue_id for d in missing_data):
                    missing_data.append({
                        'tissue_id': tissue_id,
                        'tissue_name': tissue_name,
                        'test_cells': test_tissue_counts[tissue_id]
                    })
    
    missing_df = pd.DataFrame(missing_data).sort_values('test_cells', ascending=False)
    
    output_file = Path('../analysis/missing_tissues.csv')
    output_file.parent.mkdir(exist_ok=True)
    missing_df.to_csv(output_file, index=False)
    
    print(f"Exported missing tissues to: {output_file}")
    print(f"\nPreview:")
    print(missing_df.to_string(index=False))

Exported missing tissues to: ../analysis/missing_tissues.csv

Preview:
     tissue_id                       tissue_name  test_cells
UBERON:0002686                     angular gyrus       22033
UBERON:0001111                intercostal muscle       11806
UBERON:8480009          tendon of semitendinosus        8991
UBERON:0014614 cervical spinal cord white matter        5347
UBERON:0002728                 entorhinal cortex        4991
UBERON:0002317        white matter of cerebellum        4974
UBERON:0013535            Brodmann (1909) area 4        3364
UBERON:0002185                          bronchus        3103
UBERON:0008952           upper lobe of left lung        2170
UBERON:0008953           lower lobe of left lung        1765
UBERON:0003544                brain white matter        1517
UBERON:0003126                           trachea         717


## 8. Find Tissues with No Organ/System Mapping

Analyze which tissues have NO mapping to curated organs or systems via ancestry.

In [41]:
# For each missing tissue, check if it maps to ANY curated organ/system via ancestors
print("Analyzing ancestor mappings for missing tissues...")
print("=" * 100)

# Combine all missing tissues from training and test
all_missing_tissues = training_not_in_encoder | test_not_in_encoder

# Track mapping results
tissues_with_no_mapping = []
tissues_with_organ_only = []
tissues_with_system_only = []
tissues_with_both = []

# Track all ancestors encountered (for candidate identification)
all_ancestors_counter = Counter()

for tissue_id in tqdm(sorted(all_missing_tissues), desc="Checking ancestor mappings"):
    try:
        # Get all ancestors for this tissue
        ancestors = encoder.ontology_parser.get_term_ancestors(tissue_id, include_self=True)
        
        # Track ALL ancestors for frequency analysis
        all_ancestors_counter.update(ancestors)
        
        # Find which curated organs/systems are in ancestors
        matching_organs = [org for org in ancestors if org in encoder.curated_organs_set]
        matching_systems = [sys for sys in ancestors if sys in encoder.curated_systems_set]
        
        # Get cell counts
        training_count = training_tissue_counts.get(tissue_id, 0)
        test_count = test_tissue_counts.get(tissue_id, 0)
        total_count = training_count + test_count
        
        tissue_info = {
            'tissue_id': tissue_id,
            'training_cells': training_count,
            'test_cells': test_count,
            'total_cells': total_count,
            'organ_ancestors': matching_organs,
            'system_ancestors': matching_systems
        }
        
        # Categorize based on mappings
        if not matching_organs and not matching_systems:
            tissues_with_no_mapping.append(tissue_info)
        elif matching_organs and not matching_systems:
            tissues_with_organ_only.append(tissue_info)
        elif not matching_organs and matching_systems:
            tissues_with_system_only.append(tissue_info)
        else:
            tissues_with_both.append(tissue_info)
            
    except Exception as e:
        print(f"Error processing {tissue_id}: {e}")
        tissues_with_no_mapping.append({
            'tissue_id': tissue_id,
            'training_cells': training_tissue_counts.get(tissue_id, 0),
            'test_cells': test_tissue_counts.get(tissue_id, 0),
            'total_cells': training_tissue_counts.get(tissue_id, 0) + test_tissue_counts.get(tissue_id, 0),
            'organ_ancestors': [],
            'system_ancestors': []
        })

print("\n" + "=" * 100)
print("MAPPING RESULTS")
print("=" * 100)
print(f"  Tissues with NO mapping (neither organ nor system): {len(tissues_with_no_mapping)}")
print(f"  Tissues with organ-only mapping: {len(tissues_with_organ_only)}")
print(f"  Tissues with system-only mapping: {len(tissues_with_system_only)}")
print(f"  Tissues with both organ and system mapping: {len(tissues_with_both)}")
print(f"\nTotal missing tissues analyzed: {len(all_missing_tissues)}")

Analyzing ancestor mappings for missing tissues...


Checking ancestor mappings: 100%|██████████| 201/201 [00:00<00:00, 84212.88it/s]


MAPPING RESULTS
  Tissues with NO mapping (neither organ nor system): 22
  Tissues with organ-only mapping: 10
  Tissues with system-only mapping: 27
  Tissues with both organ and system mapping: 142

Total missing tissues analyzed: 201


In [42]:
# Show details of tissues with NO mapping
if tissues_with_no_mapping:
    print("\n" + "=" * 100)
    print("CRITICAL: TISSUES WITH NO ORGAN/SYSTEM MAPPING")
    print("=" * 100)
    print("These tissues cannot map to ANY curated organ or system via ancestry.")
    print("Options:")
    print("  1. Add ancestors of these tissues to curated lists")
    print("  2. Filter these tissues from training/test data")
    print("  3. Investigate if these are data quality issues (e.g., CL terms instead of UBERON)")
    print("\nTop 20 by cell count:")
    
    # Sort by total cells
    sorted_no_mapping = sorted(tissues_with_no_mapping, key=lambda x: x['total_cells'], reverse=True)
    
    for i, tissue in enumerate(sorted_no_mapping[:20], 1):
        print(f"{i:2d}. {tissue['tissue_id']:20s} "
              f"Training: {tissue['training_cells']:8,} | "
              f"Test: {tissue['test_cells']:8,} | "
              f"Total: {tissue['total_cells']:8,}")
        
    # Calculate impact
    total_cells_no_mapping = sum(t['total_cells'] for t in tissues_with_no_mapping)
    total_missing_cells = sum(training_tissue_counts[t] for t in all_missing_tissues) + sum(test_tissue_counts[t] for t in all_missing_tissues)
    print(f"\nTotal cells affected: {total_cells_no_mapping:,}")
    print(f"Percentage of all missing tissue cells: {total_cells_no_mapping / total_missing_cells * 100:.1f}%")
else:
    print("\n✓ All missing tissues can map to at least one curated organ or system!")


CRITICAL: TISSUES WITH NO ORGAN/SYSTEM MAPPING
These tissues cannot map to ANY curated organ or system via ancestry.
Options:
  1. Add ancestors of these tissues to curated lists
  2. Filter these tissues from training/test data
  3. Investigate if these are data quality issues (e.g., CL terms instead of UBERON)

Top 20 by cell count:
 1. CL:0000084           Training:   30,718 | Test:        0 | Total:   30,718
 2. CL:0002328           Training:   19,164 | Test:        0 | Total:   19,164
 3. CL:0002334           Training:   15,089 | Test:        0 | Total:   15,089
 4. CL:0000082           Training:   10,071 | Test:        0 | Total:   10,071
 5. CL:0000115           Training:   10,000 | Test:        0 | Total:   10,000
 6. UBERON:8480009       Training:        0 | Test:    8,991 | Total:    8,991
 7. CL:0002335           Training:    7,594 | Test:        0 | Total:    7,594
 8. UBERON:0001040       Training:    7,253 | Test:        0 | Total:    7,253
 9. UBERON:0007650       Train

## 9. Identify Candidate Organs/Systems for Curation

Find frequently-occurring ancestors that are NOT in curated lists but could be added.

In [43]:
# Find candidates: ancestors that are NOT in curated lists but appear frequently
print("Finding candidate organs/systems for curation...")
print("=" * 100)

# Get all curated terms (convert list to set for union operation)
all_curated = set(encoder.curated_tissues) | encoder.curated_organs_set | encoder.curated_systems_set

# Find candidate ancestors (not in curated lists, but appear in ancestry)
candidate_ancestors = {}
for ancestor_id, frequency in all_ancestors_counter.items():
    if ancestor_id not in all_curated:
        # Get label for this term
        try:
            label = encoder.ontology_parser.get_term_label(ancestor_id)
        except:
            label = "(unknown)"
        
        candidate_ancestors[ancestor_id] = {
            'label': label,
            'frequency': frequency,
            'tissue_id': ancestor_id
        }

print(f"Found {len(candidate_ancestors):,} candidate ancestors not in curated lists")
print(f"Total curated terms: {len(all_curated)}")
print(f"Total ancestors encountered: {len(all_ancestors_counter)}")

# Sort by frequency (how many missing tissues have this as an ancestor)
sorted_candidates = sorted(candidate_ancestors.values(), key=lambda x: x['frequency'], reverse=True)

print("\nTop 50 candidate ancestors by frequency:")
print(f"{'Rank':<5} {'UBERON ID':<20} {'Label':<50} {'Frequency':<10}")
print("-" * 100)

for i, candidate in enumerate(sorted_candidates[:50], 1):
    print(f"{i:<5} {candidate['tissue_id']:<20} {candidate['label'][:48]:<50} {candidate['frequency']:<10}")

Finding candidate organs/systems for curation...
Found 681 candidate ancestors not in curated lists
Total curated terms: 81
Total ancestors encountered: 732

Top 50 candidate ancestors by frequency:
Rank  UBERON ID            Label                                              Frequency 
----------------------------------------------------------------------------------------------------
1     UBERON:0001062       anatomical entity                                  193       
2     UBERON:0000061       anatomical structure                               192       
3     UBERON:0000465       material anatomical entity                         192       
4     UBERON:0010000       multicellular anatomical structure                 191       
5     UBERON:0000468       multicellular organism                             191       
6     UBERON:0000467       anatomical system                                  184       
7     UBERON:0000062       organ                                             

In [44]:
# For each candidate, calculate total cell count coverage
print("\n" + "=" * 100)
print("CANDIDATE COVERAGE ANALYSIS")
print("=" * 100)
print("Calculating how many cells each candidate would cover if added to curated lists...")

# Map candidates to tissues that would benefit
candidate_to_tissues = {cand['tissue_id']: [] for cand in sorted_candidates[:50]}

for tissue_id in all_missing_tissues:
    try:
        ancestors = encoder.ontology_parser.get_term_ancestors(tissue_id, include_self=True)
        
        # Check which top candidates are ancestors
        for candidate_id in list(candidate_to_tissues.keys()):
            if candidate_id in ancestors:
                training_count = training_tissue_counts.get(tissue_id, 0)
                test_count = test_tissue_counts.get(tissue_id, 0)
                candidate_to_tissues[candidate_id].append({
                    'tissue_id': tissue_id,
                    'training_cells': training_count,
                    'test_cells': test_count,
                    'total_cells': training_count + test_count
                })
    except:
        pass

# Add coverage info to candidates
for candidate in sorted_candidates[:50]:
    tissues = candidate_to_tissues[candidate['tissue_id']]
    total_coverage = sum(t['total_cells'] for t in tissues)
    candidate['covered_tissues'] = len(tissues)
    candidate['total_cells_covered'] = total_coverage
    candidate['example_tissues'] = [t['tissue_id'] for t in tissues[:5]]

# Sort by total cell coverage
sorted_by_coverage = sorted(sorted_candidates[:50], key=lambda x: x['total_cells_covered'], reverse=True)

print("\nTop 30 candidates by cell coverage:")
print(f"{'Rank':<5} {'UBERON ID':<20} {'Label':<40} {'Tissues':<10} {'Cells':<12}")
print("-" * 100)

for i, candidate in enumerate(sorted_by_coverage[:30], 1):
    print(f"{i:<5} {candidate['tissue_id']:<20} {candidate['label'][:38]:<40} "
          f"{candidate['covered_tissues']:<10} {candidate['total_cells_covered']:>11,}")


CANDIDATE COVERAGE ANALYSIS
Calculating how many cells each candidate would cover if added to curated lists...

Top 30 candidates by cell coverage:
Rank  UBERON ID            Label                                    Tissues    Cells       
----------------------------------------------------------------------------------------------------
1     UBERON:0001062       anatomical entity                        193          2,256,453
2     UBERON:0000061       anatomical structure                     192          2,255,871
3     UBERON:0000465       material anatomical entity               192          2,255,871
4     UBERON:0010000       multicellular anatomical structure       191          2,249,180
5     UBERON:0000468       multicellular organism                   191          2,249,180
6     UBERON:0000467       anatomical system                        184          2,224,086
7     UBERON:0000062       organ                                    177          2,189,769
8     UBERON:0000064 

## 10. Categorize Candidates by Hierarchy Level

Determine which candidates are organ-like vs system-like based on their position in the ontology.

In [45]:
# Categorize candidates by their relationship to existing curated terms
# Heuristic: check how many descendants each candidate has
print("\n" + "=" * 100)
print("CATEGORIZING CANDIDATES")
print("=" * 100)
print("Estimating hierarchy level (organ-like vs system-like) for top candidates...")

candidate_hierarchy_info = []

for candidate in sorted_by_coverage[:30]:
    candidate_id = candidate['tissue_id']
    
    # Count how many of our missing tissues are descendants
    num_descendants = candidate['covered_tissues']
    
    # Rough heuristic:
    # - Systems typically have many descendants (>20)
    # - Organs typically have moderate descendants (5-20)
    # - Tissues typically have few descendants (<5)
    
    if num_descendants >= 20:
        category = "System-like"
    elif num_descendants >= 5:
        category = "Organ-like"
    else:
        category = "Tissue-like"
    
    candidate_hierarchy_info.append({
        **candidate,
        'category': category,
        'num_descendants': num_descendants
    })

# Group by category
print("\nSYSTEM-LIKE CANDIDATES (20+ descendant tissues):")
print(f"{'UBERON ID':<20} {'Label':<40} {'Descendants':<12} {'Cells':<12}")
print("-" * 100)
for c in candidate_hierarchy_info:
    if c['category'] == "System-like":
        print(f"{c['tissue_id']:<20} {c['label'][:38]:<40} {c['num_descendants']:<12} {c['total_cells_covered']:>11,}")

print("\n\nORGAN-LIKE CANDIDATES (5-19 descendant tissues):")
print(f"{'UBERON ID':<20} {'Label':<40} {'Descendants':<12} {'Cells':<12}")
print("-" * 100)
for c in candidate_hierarchy_info:
    if c['category'] == "Organ-like":
        print(f"{c['tissue_id']:<20} {c['label'][:38]:<40} {c['num_descendants']:<12} {c['total_cells_covered']:>11,}")

print("\n\nTISSUE-LIKE CANDIDATES (1-4 descendant tissues):")
print(f"{'UBERON ID':<20} {'Label':<40} {'Descendants':<12} {'Cells':<12}")
print("-" * 100)
for c in candidate_hierarchy_info:
    if c['category'] == "Tissue-like":
        print(f"{c['tissue_id']:<20} {c['label'][:38]:<40} {c['num_descendants']:<12} {c['total_cells_covered']:>11,}")


CATEGORIZING CANDIDATES
Estimating hierarchy level (organ-like vs system-like) for top candidates...

SYSTEM-LIKE CANDIDATES (20+ descendant tissues):
UBERON ID            Label                                    Descendants  Cells       
----------------------------------------------------------------------------------------------------
UBERON:0001062       anatomical entity                        193            2,256,453
UBERON:0000061       anatomical structure                     192            2,255,871
UBERON:0000465       material anatomical entity               192            2,255,871
UBERON:0010000       multicellular anatomical structure       191            2,249,180
UBERON:0000468       multicellular organism                   191            2,249,180
UBERON:0000467       anatomical system                        184            2,224,086
UBERON:0000062       organ                                    177            2,189,769
UBERON:0000064       organ part                   

## 11. Export Curation Report

Export comprehensive data for manual curation review.

In [46]:
# Export all candidate data to CSV for manual review
output_dir = Path('../analysis')
output_dir.mkdir(exist_ok=True)

# 1. Export candidate organs/systems
candidates_df = pd.DataFrame([{
    'uberon_id': c['tissue_id'],
    'label': c['label'],
    'category': c['category'],
    'num_descendant_tissues': c['num_descendants'],
    'total_cells_covered': c['total_cells_covered'],
    'example_tissues': '|'.join(c['example_tissues'][:5])
} for c in candidate_hierarchy_info])

candidates_file = output_dir / 'candidate_organs_systems.csv'
candidates_df.to_csv(candidates_file, index=False)
print(f"Exported {len(candidates_df)} candidates to: {candidates_file}")

# 2. Export tissues with NO mapping (with names)
no_mapping_with_names = []
for t in sorted_no_mapping:
    tissue_id = t['tissue_id']
    # Get label from ontology parser
    try:
        label = encoder.ontology_parser.get_term_label(tissue_id)
    except:
        label = "(unknown)"
    
    no_mapping_with_names.append({
        'uberon_id': tissue_id,
        'label': label,
        'training_cells': t['training_cells'],
        'test_cells': t['test_cells'],
        'total_cells': t['total_cells']
    })

no_mapping_df = pd.DataFrame(no_mapping_with_names)
no_mapping_file = output_dir / 'tissues_with_no_mapping.csv'
no_mapping_df.to_csv(no_mapping_file, index=False)
print(f"Exported {len(no_mapping_df)} tissues with no mapping to: {no_mapping_file}")

# 3. Export mapping statistics summary
mapping_summary = {
    'Total missing tissues': len(all_missing_tissues),
    'Tissues with no mapping': len(tissues_with_no_mapping),
    'Tissues with organ only': len(tissues_with_organ_only),
    'Tissues with system only': len(tissues_with_system_only),
    'Tissues with both organ and system': len(tissues_with_both),
    'Total cells in missing tissues': sum(t['total_cells'] for t in tissues_with_no_mapping + tissues_with_organ_only + tissues_with_system_only + tissues_with_both),
    'Cells with no mapping': sum(t['total_cells'] for t in tissues_with_no_mapping)
}

summary_df = pd.DataFrame([mapping_summary]).T
summary_df.columns = ['Count']
summary_file = output_dir / 'mapping_summary.csv'
summary_df.to_csv(summary_file)
print(f"Exported mapping summary to: {summary_file}")

print("\n" + "=" * 100)
print("CURATION REPORT COMPLETE")
print("=" * 100)
print(f"\nReview these files to decide which candidates to add to curated lists:")
print(f"  1. {candidates_file}")
print(f"  2. {no_mapping_file}")
print(f"  3. {summary_file}")
print(f"\nNext steps:")
print(f"  - Review system-like candidates (20+ descendants) for addition to curated_systems")
print(f"  - Review organ-like candidates (5-19 descendants) for addition to curated_organs")
print(f"  - Investigate tissues with no mapping to determine if they need special handling")

Exported 30 candidates to: ../analysis/candidate_organs_systems.csv
Exported 22 tissues with no mapping to: ../analysis/tissues_with_no_mapping.csv
Exported mapping summary to: ../analysis/mapping_summary.csv

CURATION REPORT COMPLETE

Review these files to decide which candidates to add to curated lists:
  1. ../analysis/candidate_organs_systems.csv
  2. ../analysis/tissues_with_no_mapping.csv
  3. ../analysis/mapping_summary.csv

Next steps:
  - Review system-like candidates (20+ descendants) for addition to curated_systems
  - Review organ-like candidates (5-19 descendants) for addition to curated_organs
  - Investigate tissues with no mapping to determine if they need special handling


In [47]:
# Find CL terms in tissues with no mapping
cl_terms = [t for t in tissues_with_no_mapping if t['tissue_id'].startswith('CL:')]

print(f"Found {len(cl_terms)} CL (Cell Ontology) terms with no mapping")
print("=" * 100)

if cl_terms:
    print("\nCL terms are cell types, not tissues. Looking at actual 'tissue' column values...")
    print("This will help us map CL IDs to appropriate UBERON tissue terms.\n")
    
    # Use DuckDB to find what tissues are associated with each CL term
    con = duckdb.connect(':memory:')
    
    cl_ids_list = [t['tissue_id'] for t in cl_terms]
    
    # Query both training and test data
    training_query = f"""
    SELECT 
        tissue_ontology_term_id as cl_id,
        tissue as tissue_name,
        COUNT(*) as cell_count
    FROM read_parquet('{training_dir}/*.parquet')
    WHERE tissue_ontology_term_id IN ({','.join([f"'{cl_id}'" for cl_id in cl_ids_list])})
    GROUP BY tissue_ontology_term_id, tissue
    ORDER BY tissue_ontology_term_id, cell_count DESC
    """
    
    test_query = f"""
    SELECT 
        tissue_ontology_term_id as cl_id,
        tissue as tissue_name,
        COUNT(*) as cell_count
    FROM read_parquet('{test_dir}/*.parquet')
    WHERE tissue_ontology_term_id IN ({','.join([f"'{cl_id}'" for cl_id in cl_ids_list])})
    GROUP BY tissue_ontology_term_id, tissue
    ORDER BY tissue_ontology_term_id, cell_count DESC
    """
    
    training_tissue_names = con.execute(training_query).fetchdf()
    test_tissue_names = con.execute(test_query).fetchdf()
    con.close()
    
    # Combine and aggregate
    all_tissue_names = pd.concat([training_tissue_names, test_tissue_names])
    tissue_name_summary = all_tissue_names.groupby(['cl_id', 'tissue_name'])['cell_count'].sum().reset_index()
    tissue_name_summary = tissue_name_summary.sort_values(['cl_id', 'cell_count'], ascending=[True, False])
    
    # For each CL term, get top tissue names
    cl_replacements = []
    
    for t in cl_terms:
        cl_id = t['tissue_id']
        
        # Get label
        try:
            cl_label = encoder.ontology_parser.get_term_label(cl_id)
        except:
            cl_label = "(unknown)"
        
        # Get tissue names for this CL ID
        cl_tissues = tissue_name_summary[tissue_name_summary['cl_id'] == cl_id]
        
        # Get top 3 tissue names
        top_tissues = []
        for _, row in cl_tissues.head(3).iterrows():
            top_tissues.append({
                'name': row['tissue_name'],
                'count': row['cell_count']
            })
        
        cl_replacements.append({
            'cl_id': cl_id,
            'cl_label': cl_label,
            'training_cells': t['training_cells'],
            'test_cells': t['test_cells'],
            'total_cells': t['total_cells'],
            'tissue_name_1': top_tissues[0]['name'] if len(top_tissues) > 0 else '',
            'tissue_name_1_cells': top_tissues[0]['count'] if len(top_tissues) > 0 else 0,
            'tissue_name_2': top_tissues[1]['name'] if len(top_tissues) > 1 else '',
            'tissue_name_2_cells': top_tissues[1]['count'] if len(top_tissues) > 1 else 0,
            'tissue_name_3': top_tissues[2]['name'] if len(top_tissues) > 2 else '',
            'tissue_name_3_cells': top_tissues[2]['count'] if len(top_tissues) > 2 else 0,
            'suggested_uberon_id': '',  # Manual curation needed
            'notes': 'Manual review: Map tissue names to UBERON IDs from CZ slim list'
        })
    
    # Create DataFrame and export
    cl_replacements_df = pd.DataFrame(cl_replacements)
    cl_replacements_df = cl_replacements_df.sort_values('total_cells', ascending=False)
    
    cl_replacements_file = output_dir / 'cl_to_uberon_suggestions.csv'
    cl_replacements_df.to_csv(cl_replacements_file, index=False)
    
    print(f"\n✓ Exported {len(cl_replacements_df)} CL terms to: {cl_replacements_file}")
    
    # Show preview
    print("\nPreview (top 10 by cell count):")
    print(f"{'CL ID':<15} {'CL Label':<30} {'Cells':>10} {'Top Tissue Name':<40}")
    print("-" * 100)
    for _, row in cl_replacements_df.head(10).iterrows():
        tissue_text = f"{row['tissue_name_1']} ({row['tissue_name_1_cells']:,})"
        print(f"{row['cl_id']:<15} {row['cl_label'][:28]:<30} {row['total_cells']:>10,} {tissue_text:<40}")
    
    print("\nNext step: Manually review the tissue names and map to appropriate UBERON IDs from CZ slim list")
else:
    print("\n✓ No CL terms found in tissues with no mapping")

Found 8 CL (Cell Ontology) terms with no mapping

CL terms are cell types, not tissues. Looking at actual 'tissue' column values...
This will help us map CL IDs to appropriate UBERON tissue terms.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ Exported 8 CL terms to: ../analysis/cl_to_uberon_suggestions.csv

Preview (top 10 by cell count):
CL ID           CL Label                            Cells Top Tissue Name                         
----------------------------------------------------------------------------------------------------
CL:0000084      T cell                             30,718 T cell (30,718)                         
CL:0002328      bronchial epithelial cell          19,164 bronchial epithelial cell (19,164)      
CL:0002334      preadipocyte                       15,089 preadipocyte (15,089)                   
CL:0000082      epithelial cell of lung            10,071 epithelial cell of lung (10,071)        
CL:0000115      endothelial cell                   10,000 endothelial cell (10,000)               
CL:0002335      brown preadipocyte                  7,594 brown preadipocyte (7,594)              
CL:0002633      respiratory basal cell              3,490 respiratory basal cell (3,490)          
CL:000

In [48]:
# Export CZ curated tissue, organ, and system lists with labels

# 1. Tissues
cz_tissues = []
for tissue_id in sorted(encoder.curated_tissues):
    try:
        label = encoder.ontology_parser.get_term_label(tissue_id)
    except:
        label = "(unknown)"
    
    cz_tissues.append({
        'uberon_id': tissue_id,
        'label': label
    })

cz_tissues_df = pd.DataFrame(cz_tissues)
cz_tissues_file = output_dir / 'cz_curated_tissues.csv'
cz_tissues_df.to_csv(cz_tissues_file, index=False)
print(f"✓ Exported {len(cz_tissues_df)} CZ curated tissues to: {cz_tissues_file}")

# 2. Organs
cz_organs = []
for organ_id in sorted(encoder.curated_organs):
    try:
        label = encoder.ontology_parser.get_term_label(organ_id)
    except:
        label = "(unknown)"
    
    cz_organs.append({
        'uberon_id': organ_id,
        'label': label
    })

cz_organs_df = pd.DataFrame(cz_organs)
cz_organs_file = output_dir / 'cz_curated_organs.csv'
cz_organs_df.to_csv(cz_organs_file, index=False)
print(f"✓ Exported {len(cz_organs_df)} CZ curated organs to: {cz_organs_file}")

# 3. Systems
cz_systems = []
for system_id in sorted(encoder.curated_systems):
    try:
        label = encoder.ontology_parser.get_term_label(system_id)
    except:
        label = "(unknown)"
    
    cz_systems.append({
        'uberon_id': system_id,
        'label': label
    })

cz_systems_df = pd.DataFrame(cz_systems)
cz_systems_file = output_dir / 'cz_curated_systems.csv'
cz_systems_df.to_csv(cz_systems_file, index=False)
print(f"✓ Exported {len(cz_systems_df)} CZ curated systems to: {cz_systems_file}")

print(f"\n" + "=" * 100)
print("CZ CURATED LISTS EXPORTED")
print("=" * 100)
print(f"\nTissues (81): {cz_tissues_file}")
print(f"Organs (28): {cz_organs_file}")
print(f"Systems (17): {cz_systems_file}")

print(f"\nPreview of tissues (first 20):")
print(cz_tissues_df.to_string(index=False))

print(f"\nPreview of organs (all {len(cz_organs_df)}):")
print(cz_organs_df.to_string(index=False))

print(f"\nPreview of systems (all {len(cz_systems_df)}):")
print(cz_systems_df.to_string(index=False))

✓ Exported 81 CZ curated tissues to: ../analysis/cz_curated_tissues.csv
✓ Exported 28 CZ curated organs to: ../analysis/cz_curated_organs.csv
✓ Exported 17 CZ curated systems to: ../analysis/cz_curated_systems.csv

CZ CURATED LISTS EXPORTED

Tissues (81): ../analysis/cz_curated_tissues.csv
Organs (28): ../analysis/cz_curated_organs.csv
Systems (17): ../analysis/cz_curated_systems.csv

Preview of tissues (first 20):
     uberon_id                       label
UBERON:0000004                        nose
UBERON:0000010   peripheral nervous system
UBERON:0000014                zone of skin
UBERON:0000029                  lymph node
UBERON:0000030              lamina propria
UBERON:0000056                      ureter
UBERON:0000057                     urethra
UBERON:0000059             large intestine
UBERON:0000160                   intestine
UBERON:0000175            pleural effusion
UBERON:0000178                       blood
UBERON:0000310                      breast
UBERON:0000344        

In [49]:
# Manual mappings for CL terms (cell types -> tissues)
cl_mappings = {
    'CL:0000084': {  # T cell
        'tissue': 'UBERON:0000178',  # blood
        'organ': None,
        'systems': ['UBERON:0002390']  # hematopoietic system
    },
    'CL:0002328': {  # bronchial epithelial cell
        'tissue': 'UBERON:0002048',  # lung
        'organ': 'UBERON:0002048',  # lung
        'systems': ['UBERON:0001004']  # respiratory system
    },
    'CL:0002334': {  # preadipocyte
        'tissue': 'UBERON:0001013',  # adipose tissue
        'organ': 'UBERON:0001013',  # adipose tissue
        'systems': []
    },
    'CL:0000082': {  # epithelial cell of lung
        'tissue': 'UBERON:0002048',  # lung
        'organ': 'UBERON:0002048',  # lung
        'systems': ['UBERON:0001004']  # respiratory system
    },
    'CL:0000115': {  # endothelial cell
        'tissue': 'UBERON:0002049',  # vasculature
        'organ': None,
        'systems': ['UBERON:0001009', 'UBERON:0007798']  # circulatory system; cardiovascular system
    },
    'CL:0002335': {  # brown preadipocyte
        'tissue': 'UBERON:0001348',  # brown adipose tissue
        'organ': 'UBERON:0001013',  # adipose tissue
        'systems': []
    },
    'CL:0002633': {  # respiratory basal cell
        'tissue': 'UBERON:0002048',  # lung
        'organ': 'UBERON:0002048',  # lung
        'systems': ['UBERON:0001004']  # respiratory system
    },
    'CL:0000351': {  # trophoblast cell
        'tissue': 'UBERON:0001987',  # placenta
        'organ': 'UBERON:0001987',  # placenta
        'systems': ['UBERON:0000990']  # reproductive system
    },
}

# Manual mappings for missing UBERON terms
uberon_mappings = {
    'UBERON:8480009': {  # tendon of semitendinosus
        'organ': None,
        'systems': ['UBERON:0000383', 'UBERON:0001434']  # musculature of body; skeletal system
    },
    'UBERON:0001040': {  # yolk sac
        'organ': None,
        'systems': ['UBERON:0000922']  # embryo
    },
    'UBERON:0007650': {  # esophagogastric junction
        'organ': 'UBERON:0001043',  # esophagus
        'systems': ['UBERON:0001007']  # digestive system
    },
    'UBERON:0002103': {  # hindlimb
        'organ': None,
        'systems': ['UBERON:0000383', 'UBERON:0001434']  # musculature of body; skeletal system
    },
    'UBERON:0001851': {  # cortex
        'organ': 'UBERON:0000955',  # brain
        'systems': ['UBERON:0001017', 'UBERON:0001016']  # central nervous system; nervous system
    },
    'UBERON:0016435': {  # chest wall
        'organ': None,
        'systems': ['UBERON:0000383']  # musculature of body
    },
    'UBERON:0000403': {  # scalp
        'organ': 'UBERON:0002097',  # skin of body
        'systems': []
    },
    'UBERON:0002358': {  # peritoneum
        'organ': None,
        'systems': ['UBERON:0001007']  # digestive system
    },
    'UBERON:0007795': {  # ascitic fluid
        'organ': None,
        'systems': ['UBERON:0001007']  # digestive system
    },
    'UBERON:0002102': {  # forelimb
        'organ': None,
        'systems': ['UBERON:0000383', 'UBERON:0001434']  # musculature of body; skeletal system
    },
    'UBERON:0035210': {  # paracolic gutter
        'organ': None,
        'systems': ['UBERON:0001007']  # digestive system
    },
    'UBERON:0000033': {  # head
        'organ': None,
        'systems': ['UBERON:0001016', 'UBERON:0001032']  # nervous system; sensory system
    },
    'UBERON:0001366': {  # parietal peritoneum
        'organ': None,
        'systems': ['UBERON:0001007']  # digestive system
    },
    'UBERON:0003697': {  # abdominal wall
        'organ': None,
        'systems': ['UBERON:0000383']  # musculature of body
    },
}

# Create mapping export
all_mappings = []

# Add CL mappings
for cl_id, mapping in cl_mappings.items():
    try:
        label = encoder.ontology_parser.get_term_label(cl_id)
    except:
        label = "(unknown)"
    
    all_mappings.append({
        'original_id': cl_id,
        'original_label': label,
        'type': 'CL_to_UBERON',
        'tissue_id': mapping['tissue'],
        'tissue_label': encoder.ontology_parser.get_term_label(mapping['tissue']) if mapping['tissue'] else '',
        'organ_id': mapping['organ'] if mapping['organ'] else '',
        'organ_label': encoder.ontology_parser.get_term_label(mapping['organ']) if mapping['organ'] else '',
        'system_ids': '|'.join(mapping['systems']) if mapping['systems'] else '',
        'system_labels': '|'.join([encoder.ontology_parser.get_term_label(s) for s in mapping['systems']]) if mapping['systems'] else ''
    })

# Add UBERON mappings (tissue stays same, just add organ/system)
for uberon_id, mapping in uberon_mappings.items():
    try:
        label = encoder.ontology_parser.get_term_label(uberon_id)
    except:
        label = "(unknown)"
    
    all_mappings.append({
        'original_id': uberon_id,
        'original_label': label,
        'type': 'UBERON_mapping',
        'tissue_id': uberon_id,  # Tissue stays the same
        'tissue_label': label,
        'organ_id': mapping['organ'] if mapping['organ'] else '',
        'organ_label': encoder.ontology_parser.get_term_label(mapping['organ']) if mapping['organ'] else '',
        'system_ids': '|'.join(mapping['systems']) if mapping['systems'] else '',
        'system_labels': '|'.join([encoder.ontology_parser.get_term_label(s) for s in mapping['systems']]) if mapping['systems'] else ''
    })

mappings_df = pd.DataFrame(all_mappings)
mappings_file = output_dir / 'tissue_mappings_curated.csv'
mappings_df.to_csv(mappings_file, index=False)

print(f"✓ Exported {len(mappings_df)} curated mappings to: {mappings_file}")
print(f"\n{len([m for m in all_mappings if m['type'] == 'CL_to_UBERON'])} CL → UBERON mappings")
print(f"{len([m for m in all_mappings if m['type'] == 'UBERON_mapping'])} UBERON organ/system mappings")

print("\nPreview:")
print(mappings_df.to_string(index=False))

✓ Exported 22 curated mappings to: ../analysis/tissue_mappings_curated.csv

8 CL → UBERON mappings
14 UBERON organ/system mappings

Preview:
   original_id            original_label           type      tissue_id             tissue_label       organ_id    organ_label                    system_ids                         system_labels
    CL:0000084                    T cell   CL_to_UBERON UBERON:0000178                    blood                                              UBERON:0002390                  hematopoietic system
    CL:0002328 bronchial epithelial cell   CL_to_UBERON UBERON:0002048                     lung UBERON:0002048           lung                UBERON:0001004                    respiratory system
    CL:0002334              preadipocyte   CL_to_UBERON UBERON:0001013           adipose tissue UBERON:0001013 adipose tissue                                                                    
    CL:0000082   epithelial cell of lung   CL_to_UBERON UBERON:0002048             

In [50]:
# Generate complete fallback mapping for ALL missing tissues
print("Generating complete fallback mapping system...")
print("=" * 100)

# Combine manual mappings into a lookup dict
manual_tissue_map = {}  # original_id -> tissue_id
manual_organ_map = {}   # original_id -> organ_id (or None)
manual_system_map = {}  # original_id -> [system_ids]

# Add CL mappings
for cl_id, mapping in cl_mappings.items():
    manual_tissue_map[cl_id] = mapping['tissue']
    manual_organ_map[cl_id] = mapping['organ']
    manual_system_map[cl_id] = mapping['systems']

# Add UBERON mappings (tissue stays same)
for uberon_id, mapping in uberon_mappings.items():
    manual_tissue_map[uberon_id] = uberon_id  # Keep same tissue
    manual_organ_map[uberon_id] = mapping['organ']
    manual_system_map[uberon_id] = mapping['systems']

# Now process ALL missing tissues
complete_mappings = []

for tissue in all_missing_tissues:
    tissue_id = tissue
    
    # Get label
    try:
        tissue_label = encoder.ontology_parser.get_term_label(tissue_id)
    except:
        tissue_label = "(unknown)"
    
    # Get cell counts
    training_count = training_tissue_counts.get(tissue_id, 0)
    test_count = test_tissue_counts.get(tissue_id, 0)
    total_count = training_count + test_count
    
    # Check if we have a manual mapping
    if tissue_id in manual_tissue_map:
        mapped_tissue = manual_tissue_map[tissue_id]
        mapped_organ = manual_organ_map[tissue_id]
        mapped_systems = manual_system_map[tissue_id]
        mapping_method = 'manual'
    else:
        # Use ontology hierarchy
        try:
            ancestors = encoder.ontology_parser.get_term_ancestors(tissue_id, include_self=False)
            
            # Find curated organs and systems in ancestors
            matching_organs = [org for org in ancestors if org in encoder.curated_organs_set]
            matching_systems = [sys for sys in ancestors if sys in encoder.curated_systems_set]
            
            # Keep the tissue ID as-is (not in curated list, but has organ/system mapping)
            mapped_tissue = tissue_id
            mapped_organ = matching_organs[0] if matching_organs else None
            mapped_systems = matching_systems
            mapping_method = 'ontology_hierarchy'
            
        except Exception as e:
            # Fallback: no mapping possible
            mapped_tissue = tissue_id
            mapped_organ = None
            mapped_systems = []
            mapping_method = 'none (error or no ancestors)'
    
    # Get labels
    try:
        mapped_tissue_label = encoder.ontology_parser.get_term_label(mapped_tissue)
    except:
        mapped_tissue_label = tissue_label
    
    organ_label = ''
    if mapped_organ:
        try:
            organ_label = encoder.ontology_parser.get_term_label(mapped_organ)
        except:
            organ_label = '(unknown)'
    
    system_labels = []
    for sys_id in mapped_systems:
        try:
            system_labels.append(encoder.ontology_parser.get_term_label(sys_id))
        except:
            system_labels.append('(unknown)')
    
    complete_mappings.append({
        'original_id': tissue_id,
        'original_label': tissue_label,
        'training_cells': training_count,
        'test_cells': test_count,
        'total_cells': total_count,
        'mapping_method': mapping_method,
        'mapped_tissue_id': mapped_tissue,
        'mapped_tissue_label': mapped_tissue_label,
        'mapped_organ_id': mapped_organ if mapped_organ else '',
        'mapped_organ_label': organ_label,
        'mapped_system_ids': '|'.join(mapped_systems) if mapped_systems else '',
        'mapped_system_labels': '|'.join(system_labels) if system_labels else '',
        'in_cz_tissue_list': 'yes' if mapped_tissue in encoder.curated_tissues else 'no',
        'has_organ_mapping': 'yes' if mapped_organ else 'no',
        'has_system_mapping': 'yes' if mapped_systems else 'no'
    })

# Create DataFrame
complete_mappings_df = pd.DataFrame(complete_mappings)
complete_mappings_df = complete_mappings_df.sort_values('total_cells', ascending=False)

# Export
complete_mappings_file = output_dir / 'complete_tissue_fallback_mappings.csv'
complete_mappings_df.to_csv(complete_mappings_file, index=False)

print(f"\n✓ Exported {len(complete_mappings_df)} complete tissue mappings to: {complete_mappings_file}")

# Summary statistics
print("\n" + "=" * 100)
print("MAPPING SUMMARY")
print("=" * 100)

method_counts = complete_mappings_df['mapping_method'].value_counts()
print(f"\nMapping methods:")
for method, count in method_counts.items():
    print(f"  {method}: {count} tissues")

coverage_stats = {
    'Has tissue in CZ list': (complete_mappings_df['in_cz_tissue_list'] == 'yes').sum(),
    'Has organ mapping': (complete_mappings_df['has_organ_mapping'] == 'yes').sum(),
    'Has system mapping': (complete_mappings_df['has_system_mapping'] == 'yes').sum(),
    'Fully mapped (tissue + organ + system)': ((complete_mappings_df['in_cz_tissue_list'] == 'yes') & 
                                                 (complete_mappings_df['has_organ_mapping'] == 'yes') & 
                                                 (complete_mappings_df['has_system_mapping'] == 'yes')).sum()
}

print(f"\nCoverage statistics:")
for stat, count in coverage_stats.items():
    pct = count / len(complete_mappings_df) * 100
    print(f"  {stat}: {count}/{len(complete_mappings_df)} ({pct:.1f}%)")

# Cell coverage
total_cells_mapped = complete_mappings_df['total_cells'].sum()
cells_with_system = complete_mappings_df[complete_mappings_df['has_system_mapping'] == 'yes']['total_cells'].sum()
cells_with_organ = complete_mappings_df[complete_mappings_df['has_organ_mapping'] == 'yes']['total_cells'].sum()

print(f"\nCell coverage:")
print(f"  Total cells with missing tissues: {total_cells_mapped:,}")
print(f"  Cells with system mapping: {cells_with_system:,} ({cells_with_system/total_cells_mapped*100:.1f}%)")
print(f"  Cells with organ mapping: {cells_with_organ:,} ({cells_with_organ/total_cells_mapped*100:.1f}%)")

print("\n" + "=" * 100)
print("Preview (top 20 by cell count):")
print("=" * 100)
preview_cols = ['original_id', 'original_label', 'total_cells', 'mapping_method', 
                'mapped_tissue_label', 'mapped_organ_label', 'mapped_system_labels']
print(complete_mappings_df[preview_cols].head(20).to_string(index=False))

Generating complete fallback mapping system...

✓ Exported 201 complete tissue mappings to: ../analysis/complete_tissue_fallback_mappings.csv

MAPPING SUMMARY

Mapping methods:
  ontology_hierarchy: 179 tissues
  manual: 22 tissues

Coverage statistics:
  Has tissue in CZ list: 8/201 (4.0%)
  Has organ mapping: 161/201 (80.1%)
  Has system mapping: 188/201 (93.5%)
  Fully mapped (tissue + organ + system): 4/201 (2.0%)

Cell coverage:
  Total cells with missing tissues: 2,355,913
  Cells with system mapping: 2,291,501 (97.3%)
  Cells with organ mapping: 1,863,642 (79.1%)

Preview (top 20 by cell count):
   original_id                 original_label  total_cells     mapping_method            mapped_tissue_label mapped_organ_label                              mapped_system_labels
UBERON:0002116                          ileum        95260 ontology_hierarchy                          ileum          intestine                                  digestive system
UBERON:0000991                    

In [51]:
# Check each tissue in training+test data to see if it can find a CZ slim ancestor
from cellxgene_ontology_guide.ontology_parser import OntologyParser

ontology_parser = OntologyParser()

# Get all unique tissues from training and test
all_tissues = training_tissue_df['tissue_ontology_term_id'].unique().tolist() + \
              test_tissue_df['tissue_ontology_term_id'].unique().tolist()
all_tissues = list(set(all_tissues))

# Get CZ slim curated tissues
curated_tissues_set = set(encoder.curated_tissues)
curated_organs_set = encoder.curated_organs_set
curated_systems_set = encoder.curated_systems_set

print(f"Total unique tissues in data: {len(all_tissues)}")
print(f"CZ slim curated tissues: {len(curated_tissues_set)}")
print()

# Analyze each tissue
tissue_analysis = []

for tissue_id in all_tissues:
    result = {
        'tissue_id': tissue_id,
        'is_curated': tissue_id in curated_tissues_set,
        'has_cz_tissue_ancestor': False,
        'nearest_tissue_ancestor': None,
        'has_cz_organ_ancestor': False,
        'has_cz_system_ancestor': False,
        'organ_ancestors': [],
        'system_ancestors': [],
        'needs_manual_fallback': False,
    }
    
    # If already curated, skip
    if result['is_curated']:
        tissue_analysis.append(result)
        continue
    
    # Get all ancestors
    try:
        ancestors = ontology_parser.get_term_ancestors(tissue_id, include_self=False)
        
        # Check for CZ slim tissue ancestor
        for ancestor in ancestors:
            if ancestor in curated_tissues_set:
                result['has_cz_tissue_ancestor'] = True
                result['nearest_tissue_ancestor'] = ancestor
                break
        
        # Check for organ/system ancestors
        result['organ_ancestors'] = [a for a in ancestors if a in curated_organs_set]
        result['system_ancestors'] = [a for a in ancestors if a in curated_systems_set]
        result['has_cz_organ_ancestor'] = len(result['organ_ancestors']) > 0
        result['has_cz_system_ancestor'] = len(result['system_ancestors']) > 0
        
    except Exception as e:
        # Invalid tissue ID or other error
        result['error'] = str(e)
    
    # Determine if manual fallback is needed
    # Manual fallback is needed if:
    # - No CZ slim tissue ancestor AND
    # - No CZ slim organ/system ancestor
    if not result['has_cz_tissue_ancestor'] and \
       not result['has_cz_organ_ancestor'] and \
       not result['has_cz_system_ancestor']:
        result['needs_manual_fallback'] = True
    
    tissue_analysis.append(result)

# Convert to DataFrame for analysis
analysis_df = pd.DataFrame(tissue_analysis)

print("Tissue Coverage Analysis:")
print(f"  Already curated (CZ slim): {analysis_df['is_curated'].sum()}")
print(f"  Can use CZ slim tissue ancestor: {analysis_df['has_cz_tissue_ancestor'].sum()}")
print(f"  Can use CZ slim organ/system ancestor: {(analysis_df['has_cz_organ_ancestor'] | analysis_df['has_cz_system_ancestor']).sum()}")
print(f"  Needs manual fallback: {analysis_df['needs_manual_fallback'].sum()}")
print()

# Show tissues that need manual fallback
manual_fallback_tissues = analysis_df[analysis_df['needs_manual_fallback']]
print(f"Tissues requiring manual fallback ({len(manual_fallback_tissues)}):")
for _, row in manual_fallback_tissues.iterrows():
    try:
        label = ontology_parser.get_term_label(row['tissue_id'])
        print(f"  {row['tissue_id']}: {label}")
    except:
        print(f"  {row['tissue_id']}: <unknown>")

analysis_df

Total unique tissues in data: 239
CZ slim curated tissues: 81

Tissue Coverage Analysis:
  Already curated (CZ slim): 38
  Can use CZ slim tissue ancestor: 183
  Can use CZ slim organ/system ancestor: 179
  Needs manual fallback: 18

Tissues requiring manual fallback (18):
  UBERON:0001851: cortex
  UBERON:0016435: chest wall
  UBERON:0002103: hindlimb
  CL:0000351: trophoblast cell
  UBERON:0035210: paracolic gutter
  UBERON:8480009: tendon of semitendinosus
  CL:0002633: respiratory basal cell
  UBERON:0000033: head
  UBERON:0002102: forelimb
  CL:0000115: endothelial cell
  CL:0000082: epithelial cell of lung
  CL:0002328: bronchial epithelial cell
  CL:0002334: preadipocyte
  UBERON:0007650: esophagogastric junction
  CL:0002335: brown preadipocyte
  UBERON:0001040: yolk sac
  CL:0000084: T cell
  UBERON:0000403: scalp


,tissue_id,is_curated,has_cz_tissue_ancestor,nearest_tissue_ancestor,has_cz_organ_ancestor,has_cz_system_ancestor,organ_ancestors,system_ancestors,needs_manual_fallback
0,UBERON:0002370,True,False,None,False,False,[],[],False
1,UBERON:0002371,True,False,None,False,False,[],[],False
2,UBERON:0002094,False,True,UBERON:0000948,True,True,[UBERON:0000948],"[UBERON:0004535, UBERON:0001009]",False
3,UBERON:0001851,False,False,None,False,False,[],[],True
4,UBERON:0001416,True,False,None,False,False,[],[],False
...,...,...,...,...,...,...,...,...,...
234,UBERON:0001621,False,True,UBERON:0000948,True,True,[UBERON:0000948],"[UBERON:0004535, UBERON:0001009]",False
235,UBERON:0002190,True,False,None,False,False,[],[],False
236,UBERON:0003889,True,False,None,False,False,[],[],False
237,UBERON:8440051,False,True,UBERON:0001017,True,True,[UBERON:0000955],"[UBERON:0001017, UBERON:0001016]",False


## 17. Detailed Breakdown by Mapping Strategy

Let's break down the tissues by which strategy they use:

In [52]:
# Categorize tissues by strategy
analysis_df['strategy'] = 'unknown'

# Tier 1: Curated CZ slim tissues
analysis_df.loc[analysis_df['is_curated'], 'strategy'] = 'Tier 1: Curated (CZ slim)'

# Tier 2: Has CZ slim tissue ancestor
analysis_df.loc[
    (~analysis_df['is_curated']) & analysis_df['has_cz_tissue_ancestor'], 
    'strategy'
] = 'Tier 2: CZ slim tissue ancestor'

# Tier 3a: Has CZ slim organ/system ancestor (no tissue ancestor)
analysis_df.loc[
    (~analysis_df['is_curated']) & 
    (~analysis_df['has_cz_tissue_ancestor']) & 
    (analysis_df['has_cz_organ_ancestor'] | analysis_df['has_cz_system_ancestor']),
    'strategy'
] = 'Tier 3a: CZ slim organ/system ancestor'

# Tier 3b: Needs manual fallback
analysis_df.loc[analysis_df['needs_manual_fallback'], 'strategy'] = 'Tier 3b: Manual fallback required'

# Count by strategy
strategy_counts = analysis_df['strategy'].value_counts()
print("Tissues by mapping strategy:")
for strategy, count in strategy_counts.items():
    print(f"  {strategy}: {count}")

print(f"\nTotal: {len(analysis_df)}")

# Show breakdown with cell counts
print("\n" + "="*80)
print("Cell Coverage by Strategy")
print("="*80)

# Merge with cell counts from training data
training_counts = training_tissue_df.groupby('tissue_ontology_term_id').size().reset_index(name='training_cells')
test_counts = test_tissue_df.groupby('tissue_ontology_term_id').size().reset_index(name='test_cells')

analysis_with_counts = analysis_df.merge(
    training_counts, 
    left_on='tissue_id', 
    right_on='tissue_ontology_term_id', 
    how='left'
).merge(
    test_counts,
    left_on='tissue_id',
    right_on='tissue_ontology_term_id',
    how='left'
)

analysis_with_counts['training_cells'] = analysis_with_counts['training_cells'].fillna(0).astype(int)
analysis_with_counts['test_cells'] = analysis_with_counts['test_cells'].fillna(0).astype(int)
analysis_with_counts['total_cells'] = analysis_with_counts['training_cells'] + analysis_with_counts['test_cells']

# Summary by strategy
strategy_summary = analysis_with_counts.groupby('strategy').agg({
    'tissue_id': 'count',
    'total_cells': 'sum',
    'training_cells': 'sum',
    'test_cells': 'sum'
}).rename(columns={'tissue_id': 'num_tissues'})

strategy_summary['pct_cells'] = (strategy_summary['total_cells'] / strategy_summary['total_cells'].sum() * 100).round(2)

print(strategy_summary.sort_values('total_cells', ascending=False))

analysis_with_counts

Tissues by mapping strategy:
  Tier 2: CZ slim tissue ancestor: 183
  Tier 1: Curated (CZ slim): 38
  Tier 3b: Manual fallback required: 18

Total: 239

Cell Coverage by Strategy
                                   num_tissues  total_cells  training_cells  \
strategy                                                                      
Tier 2: CZ slim tissue ancestor            183          193             182   
Tier 1: Curated (CZ slim)                   38           43              38   
Tier 3b: Manual fallback required           18           18              17   

                                   test_cells  pct_cells  
strategy                                                  
Tier 2: CZ slim tissue ancestor            11      75.98  
Tier 1: Curated (CZ slim)                   5      16.93  
Tier 3b: Manual fallback required           1       7.09  


,tissue_id,is_curated,has_cz_tissue_ancestor,nearest_tissue_ancestor,has_cz_organ_ancestor,has_cz_system_ancestor,organ_ancestors,system_ancestors,needs_manual_fallback,strategy,tissue_ontology_term_id_x,training_cells,tissue_ontology_term_id_y,test_cells,total_cells
0,UBERON:0002370,True,False,None,False,False,[],[],False,Tier 1: Curated (CZ slim),UBERON:0002370,1,NaN,0,1
1,UBERON:0002371,True,False,None,False,False,[],[],False,Tier 1: Curated (CZ slim),UBERON:0002371,1,NaN,0,1
2,UBERON:0002094,False,True,UBERON:0000948,True,True,[UBERON:0000948],"[UBERON:0004535, UBERON:0001009]",False,Tier 2: CZ slim tissue ancestor,UBERON:0002094,1,NaN,0,1
3,UBERON:0001851,False,False,None,False,False,[],[],True,Tier 3b: Manual fallback required,UBERON:0001851,1,NaN,0,1
4,UBERON:0001416,True,False,None,False,False,[],[],False,Tier 1: Curated (CZ slim),UBERON:0001416,1,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,UBERON:0001621,False,True,UBERON:0000948,True,True,[UBERON:0000948],"[UBERON:0004535, UBERON:0001009]",False,Tier 2: CZ slim tissue ancestor,UBERON:0001621,1,NaN,0,1
235,UBERON:0002190,True,False,None,False,False,[],[],False,Tier 1: Curated (CZ slim),UBERON:0002190,1,NaN,0,1
236,UBERON:0003889,True,False,None,False,False,[],[],False,Tier 1: Curated (CZ slim),UBERON:0003889,1,NaN,0,1
237,UBERON:8440051,False,True,UBERON:0001017,True,True,[UBERON:0000955],"[UBERON:0001017, UBERON:0001016]",False,Tier 2: CZ slim tissue ancestor,UBERON:8440051,1,NaN,0,1


## 18. Export Tissues Requiring Manual Fallback

Export the list of tissues that truly need manual fallback mappings (Tier 3b):

In [54]:
# Get tissues that need manual fallback
manual_fallback_df = analysis_with_counts[
    analysis_with_counts['strategy'] == 'Tier 3b: Manual fallback required'
].copy()

# Add labels
manual_fallback_df['label'] = manual_fallback_df['tissue_id'].apply(
    lambda x: ontology_parser.get_term_label(x) if ontology_parser.is_valid_term_id(x) else 'Unknown'
)

# Sort by cell count
manual_fallback_df = manual_fallback_df.sort_values('total_cells', ascending=False)

print(f"Tissues requiring manual fallback: {len(manual_fallback_df)}")
print(f"Total cells affected: {manual_fallback_df['total_cells'].sum():,}")
print(f"Percentage of total cells: {manual_fallback_df['total_cells'].sum() / analysis_with_counts['total_cells'].sum() * 100:.2f}%")
print()

# Display
print("Tissues needing manual fallback (sorted by cell count):")
for _, row in manual_fallback_df.iterrows():
    print(f"  {row['tissue_id']}: {row['label']}")
    print(f"    Cells: {row['total_cells']:,} (train: {row['training_cells']:,}, test: {row['test_cells']:,})")

# Export to CSV
output_file = output_dir / 'tissues_needing_manual_fallback.csv'
manual_fallback_df[['tissue_id', 'label', 'total_cells', 'training_cells', 'test_cells']].to_csv(
    output_file, index=False
)
print(f"\nExported to: {output_file}")

manual_fallback_df[['tissue_id', 'label', 'total_cells', 'training_cells', 'test_cells']]

Tissues requiring manual fallback: 18
Total cells affected: 18
Percentage of total cells: 7.09%

Tissues needing manual fallback (sorted by cell count):
  UBERON:0001851: cortex
    Cells: 1 (train: 1, test: 0)
  UBERON:0016435: chest wall
    Cells: 1 (train: 1, test: 0)
  UBERON:0002103: hindlimb
    Cells: 1 (train: 1, test: 0)
  CL:0000351: trophoblast cell
    Cells: 1 (train: 1, test: 0)
  UBERON:0035210: paracolic gutter
    Cells: 1 (train: 1, test: 0)
  UBERON:8480009: tendon of semitendinosus
    Cells: 1 (train: 0, test: 1)
  CL:0002633: respiratory basal cell
    Cells: 1 (train: 1, test: 0)
  UBERON:0000033: head
    Cells: 1 (train: 1, test: 0)
  UBERON:0002102: forelimb
    Cells: 1 (train: 1, test: 0)
  CL:0000115: endothelial cell
    Cells: 1 (train: 1, test: 0)
  CL:0000082: epithelial cell of lung
    Cells: 1 (train: 1, test: 0)
  CL:0002328: bronchial epithelial cell
    Cells: 1 (train: 1, test: 0)
  CL:0002334: preadipocyte
    Cells: 1 (train: 1, test: 0)
  UBE

,tissue_id,label,total_cells,training_cells,test_cells
3,UBERON:0001851,cortex,1,1,0
8,UBERON:0016435,chest wall,1,1,0
40,UBERON:0002103,hindlimb,1,1,0
41,CL:0000351,trophoblast cell,1,1,0
45,UBERON:0035210,paracolic gutter,1,1,0
64,UBERON:8480009,tendon of semitendinosus,1,0,1
77,CL:0002633,respiratory basal cell,1,1,0
78,UBERON:0000033,head,1,1,0
104,UBERON:0002102,forelimb,1,1,0
110,CL:0000115,endothelial cell,1,1,0


## 19. Verify Current Fallback Mappings

Check which of the 22 tissues in our current FALLBACK_TISSUE_MAPPINGS actually have CZ slim ancestors (and thus don't need to be in the table):

In [56]:
# Load the current FALLBACK_TISSUE_MAPPINGS from the encoder
from data_loading.tissue_encoder import FALLBACK_TISSUE_MAPPINGS

print(f"Total tissues in FALLBACK_TISSUE_MAPPINGS: {len(FALLBACK_TISSUE_MAPPINGS)}")
print()

# Check each one against our analysis
fallback_verification = []

for tissue_id in FALLBACK_TISSUE_MAPPINGS.keys():
    # Find this tissue in our analysis
    tissue_info = analysis_with_counts[analysis_with_counts['tissue_id'] == tissue_id]
    
    if len(tissue_info) > 0:
        row = tissue_info.iloc[0]
        fallback_verification.append({
            'tissue_id': tissue_id,
            'label': ontology_parser.get_term_label(tissue_id) if ontology_parser.is_valid_term_id(tissue_id) else 'Unknown',
            'strategy': row['strategy'],
            'has_cz_tissue_ancestor': row['has_cz_tissue_ancestor'],
            'nearest_tissue_ancestor': row['nearest_tissue_ancestor'],
            'has_cz_organ_ancestor': row['has_cz_organ_ancestor'],
            'has_cz_system_ancestor': row['has_cz_system_ancestor'],
            'total_cells': row['total_cells'],
            'needs_manual_fallback': row['strategy'] == 'Tier 3b: Manual fallback required'
        })
    else:
        # Not in our data
        fallback_verification.append({
            'tissue_id': tissue_id,
            'label': ontology_parser.get_term_label(tissue_id) if ontology_parser.is_valid_term_id(tissue_id) else 'Unknown',
            'strategy': 'Not in training/test data',
            'has_cz_tissue_ancestor': None,
            'nearest_tissue_ancestor': None,
            'has_cz_organ_ancestor': None,
            'has_cz_system_ancestor': None,
            'total_cells': 0,
            'needs_manual_fallback': None
        })

fallback_verification_df = pd.DataFrame(fallback_verification)

print("Current FALLBACK_TISSUE_MAPPINGS analysis:")
print(f"  Tissues with CZ slim tissue ancestor: {fallback_verification_df['has_cz_tissue_ancestor'].sum()}")
print(f"  Tissues with CZ slim organ/system ancestor (no tissue): {((fallback_verification_df['has_cz_organ_ancestor'] | fallback_verification_df['has_cz_system_ancestor']) & ~fallback_verification_df['has_cz_tissue_ancestor']).sum()}")
print(f"  Tissues needing manual fallback: {fallback_verification_df['needs_manual_fallback'].sum()}")
print(f"  Not in our dataset: {(fallback_verification_df['strategy'] == 'Not in training/test data').sum()}")
print()

print("\nTissues that DON'T need to be in FALLBACK_TISSUE_MAPPINGS (have CZ slim ancestors):")
unnecessary = fallback_verification_df[
    (fallback_verification_df['has_cz_tissue_ancestor'] == True) |
    (fallback_verification_df['has_cz_organ_ancestor'] == True) |
    (fallback_verification_df['has_cz_system_ancestor'] == True)
]
for _, row in unnecessary.iterrows():
    print(f"  {row['tissue_id']}: {row['label']}")
    print(f"    Strategy: {row['strategy']}")
    if row['has_cz_tissue_ancestor']:
        ancestor_label = ontology_parser.get_term_label(row['nearest_tissue_ancestor'])
        print(f"    → Has CZ tissue ancestor: {row['nearest_tissue_ancestor']} ({ancestor_label})")
    if row['has_cz_organ_ancestor'] or row['has_cz_system_ancestor']:
        print(f"    → Has CZ organ/system ancestor")
    print()

print("\nTissues that SHOULD be in FALLBACK_TISSUE_MAPPINGS (no CZ slim ancestors):")
necessary = fallback_verification_df[fallback_verification_df['needs_manual_fallback'] == True]
for _, row in necessary.iterrows():
    print(f"  {row['tissue_id']}: {row['label']} ({row['total_cells']:,} cells)")

fallback_verification_df

Total tissues in FALLBACK_TISSUE_MAPPINGS: 22

Current FALLBACK_TISSUE_MAPPINGS analysis:
  Tissues with CZ slim tissue ancestor: 4
  Tissues with CZ slim organ/system ancestor (no tissue): 0
  Tissues needing manual fallback: 18
  Not in our dataset: 0


Tissues that DON'T need to be in FALLBACK_TISSUE_MAPPINGS (have CZ slim ancestors):
  UBERON:0002358: peritoneum
    Strategy: Tier 2: CZ slim tissue ancestor
    → Has CZ tissue ancestor: UBERON:0000916 (abdomen)

  UBERON:0007795: ascitic fluid
    Strategy: Tier 2: CZ slim tissue ancestor
    → Has CZ tissue ancestor: UBERON:0000916 (abdomen)

  UBERON:0001366: parietal peritoneum
    Strategy: Tier 2: CZ slim tissue ancestor
    → Has CZ tissue ancestor: UBERON:0000916 (abdomen)

  UBERON:0003697: abdominal wall
    Strategy: Tier 2: CZ slim tissue ancestor
    → Has CZ tissue ancestor: UBERON:0000916 (abdomen)


Tissues that SHOULD be in FALLBACK_TISSUE_MAPPINGS (no CZ slim ancestors):
  CL:0000084: T cell (1 cells)
  CL:0002328:

,tissue_id,label,strategy,has_cz_tissue_ancestor,nearest_tissue_ancestor,has_cz_organ_ancestor,has_cz_system_ancestor,total_cells,needs_manual_fallback
0,CL:0000084,T cell,Tier 3b: Manual fallback required,False,None,False,False,1,True
1,CL:0002328,bronchial epithelial cell,Tier 3b: Manual fallback required,False,None,False,False,1,True
2,CL:0002334,preadipocyte,Tier 3b: Manual fallback required,False,None,False,False,1,True
3,CL:0000082,epithelial cell of lung,Tier 3b: Manual fallback required,False,None,False,False,1,True
4,CL:0000115,endothelial cell,Tier 3b: Manual fallback required,False,None,False,False,1,True
5,CL:0002335,brown preadipocyte,Tier 3b: Manual fallback required,False,None,False,False,1,True
6,CL:0002633,respiratory basal cell,Tier 3b: Manual fallback required,False,None,False,False,1,True
7,CL:0000351,trophoblast cell,Tier 3b: Manual fallback required,False,None,False,False,1,True
8,UBERON:8480009,tendon of semitendinosus,Tier 3b: Manual fallback required,False,None,False,False,1,True
9,UBERON:0001040,yolk sac,Tier 3b: Manual fallback required,False,None,False,False,1,True
